# Caretta Lab — ST-FMR Testing Notebook

Spin-torque ferromagnetic resonance (ST-FMR) using the PIEC measurement pipeline. This is the **local experiment notebook**, structured like `Measurements/AMR/AMR_testing.ipynb` and migrated from the uploaded `STFMR_measurement.ipynb`.

**COPY THIS NOTEBOOK to a local working directory before editing or running.** The default instrument setup is virtual. **All scan cells are gated by `RUN_SCANS = False`; change this explicitly before data acquisition.** Never enable physical magnet or RF outputs before validating calibration and instrument limits.

See `stfmr_measurement.md` for original-to-PIEC API mapping, safety, and metadata.

## 0 — Imports and resource discovery

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from piec.measurement.stfmr import STFMR, load_stfmr_data
from piec.drivers.daq.virtual_daq import VirtualDaq
from piec.drivers.lockin.virtual_lockin import VirtualLockin
from piec.drivers.rf_source.virtual_rf_source import VirtualRFSource
from piec.drivers.stepper_motor.virtual_stepper import VirtualStepper
from piec.drivers.utilities import PiecManager

pm = PiecManager()
print('VISA resources:', pm.list_resources())

## 1 — Instruments (virtual by default)

The uploaded notebook used `ekpy` Digilent `v_out/v_in`, an SRS830 lock-in, an Anritsu GPIB RF generator (`GPIB0::5::INSTR`), and `Arduino_Stepper` on `ASRL4::INSTR`. PIEC supplies standard `write_AO`, `read_AI`, `get_X_Y`, `set_frequency`, `set_power`, `output`, and `step` interfaces. The generic PIEC RFSource has a virtual implementation, but **no verified physical Anritsu implementation was present in the inspected branch**. Do not rely on raw original SCPI strings without the instrument programming manual.

For physical hardware, create `USB231('0')` or the verified DAQ model, `SRS830('GPIB0::8::INSTR')`, `Geos_Stepper('ASRL4::INSTR')`, and a verified physical RFSource driver, replacing the virtual assignments only after bench checks.

In [ ]:
# VIRTUAL ONLY: creating drivers here does not apply a physical field or RF signal.
USE_VIRTUAL = True
RUN_SCANS = False  # Must be explicitly changed to True to acquire data.

if USE_VIRTUAL:
    daq = VirtualDaq('VIRTUAL')
    lockin = VirtualLockin('VIRTUAL')
    rf_source = VirtualRFSource('VIRTUAL')
    stepper = VirtualStepper('VIRTUAL')
else:
    raise RuntimeError(
        'Physical instruments must be constructed explicitly after verifying '
        'DAQ model, Anritsu programming manual, motor direction, calibration and safe limits.'
    )

for label, obj in [('DAQ', daq), ('Lock-in', lockin),
                   ('RF source', rf_source), ('Stepper', stepper)]:
    print(label, obj.idn())

## 2 — Shared experiment configuration

Use the **same field units as your `side_center_calibration.xlsx` workbook**. The original notebook does not specify whether `center field` represents tesla, Oe, or another scale. Its `side field` column represents the commanded output voltage in V. The notebook's old angular conversion was `380 motor steps = 180 degrees` (unverified). No automatic homing or driver reset occurs when the measurement starts.

In [ ]:
SAVE_DIR = Path('scratch')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CALIBRATION_FILE = None  # Physical: Path('side_center_calibration.xlsx')
FIELD_UNIT = 'calibration units'  # Replace with the *verified* unit of the workbook.

# Demonstration factor is solely for VIRTUAL use; never use as a physical calibration.
LINEAR_VOLTS_PER_FIELD_UNIT = 1.0 if USE_VIRTUAL else None

LOCKIN_AMPLITUDE_V = 0.4
LOCKIN_FREQUENCY_HZ = 17.37
RF_FREQUENCY_HZ = 3e9
RF_POWER_DBM = 10.0
FIELD_VALUES = np.linspace(0.7, 0, 50, endpoint=False)  # matches original code
MOTOR_STEPS_PER_180_DEG = 380.0  # original notebook conversion; bench-verify

# Set short timings only for the virtual demonstration. Restore e.g. 3 s averaging
# and appropriate RF/field settling times for real hardware.
MEASURE_TIME_S = 0.0 if USE_VIRTUAL else 3.0
FIELD_SETTLE_S = 0.0 if USE_VIRTUAL else 1.0
RF_SETTLE_S = 0.0 if USE_VIRTUAL else 10.0
ANGLE_SETTLE_S = 0.0 if USE_VIRTUAL else 1.0


def make_stfmr(*, fields=FIELD_VALUES, rf_frequencies_hz=(RF_FREQUENCY_HZ,),
               rf_powers_dbm=(RF_POWER_DBM,),
               lockin_frequencies_hz=(LOCKIN_FREQUENCY_HZ,), angles_deg=(0.0,)):
    return STFMR(
        daq=daq, lockin=lockin, rf_source=rf_source, arduino=stepper,
        fields=fields, rf_frequencies_hz=rf_frequencies_hz,
        rf_powers_dbm=rf_powers_dbm,
        lockin_frequencies_hz=lockin_frequencies_hz, angles_deg=angles_deg,
        lockin_amplitude=LOCKIN_AMPLITUDE_V,
        calibration_file=CALIBRATION_FILE,
        linear_volts_per_field_unit=LINEAR_VOLTS_PER_FIELD_UNIT,
        field_unit=FIELD_UNIT, save_dir=SAVE_DIR,
        measure_time=MEASURE_TIME_S, field_settle_s=FIELD_SETTLE_S,
        rf_settle_s=RF_SETTLE_S, angle_settle_s=ANGLE_SETTLE_S,
        motor_steps_per_180_deg=MOTOR_STEPS_PER_180_DEG,
        forward_direction=0, live_plot=False,
    )

## 3 — Field dependence (original `field_dependence_measurement`)

At fixed RF frequency/power, step field from +0.7 toward 0 with 50 points and record **both X and Y**. The original loop did not include its final field endpoint, so we use `endpoint=False`. All scans shut off AO and RF output even if acquisition fails.

In [ ]:
field_experiment = make_stfmr(fields=FIELD_VALUES)
if RUN_SCANS:
    field_file = field_experiment.run_experiment()
    print('Saved:', field_file)
else:
    print('Dry setup only. Set RUN_SCANS = True to execute.')

### Reverse-field pass

Original positive/negative sweeps were saved separately. This preserves distinct branches; ensure the real calibration workbook covers the negative field range.

In [ ]:
negative_fields = np.linspace(-0.7, 0, 50, endpoint=False)
if RUN_SCANS:
    negative_experiment = make_stfmr(fields=negative_fields)
    print('Saved:', negative_experiment.run_experiment())

## 4 — RF sweeps (original `freq_sweep`, `rf_freq_scan`, `rf_power_scan`, `double_freq_sweep`)

Each scan axis takes a sequence of values, and every combination becomes an acquired row. Frequencies in the Python interface are **Hz**, powers are **dBm**. For combined sweeps, field is the innermost axis.

In [ ]:
# RF frequency vs field: 2 to 7.5 GHz, 0.25 GHz steps (as in notebook).
rf_frequencies = np.arange(2e9, 7.75e9, 0.25e9)
if RUN_SCANS:
    frequency_field = make_stfmr(rf_frequencies_hz=rf_frequencies,
                                 rf_powers_dbm=(4.0,), fields=FIELD_VALUES)
    print('Saved:', frequency_field.run_experiment())

In [ ]:
# RF power scan at constant field (original used -20 to <20 dBm in 0.5 dB steps).
# This is a broad power range: validate safe RF power before changing RUN_SCANS.
rf_powers = np.arange(-20, 20, 0.5)
if RUN_SCANS:
    power_scan = make_stfmr(fields=(0.5,), rf_frequencies_hz=(3e9,),
                            rf_powers_dbm=rf_powers)
    print('Saved:', power_scan.run_experiment())

# RF frequency scan at a constant field.
if RUN_SCANS:
    frequency_scan = make_stfmr(fields=(0.3,),
                                rf_frequencies_hz=np.arange(1e9, 10e9, 0.2e9))
    print('Saved:', frequency_scan.run_experiment())

In [ ]:
# Combined RF frequency × RF power × lock-in reference-frequency × field scan.
# Reflects original double_freq_sweep; adjust arrays to limit acquisition time.
if RUN_SCANS:
    combined_scan = make_stfmr(
        fields=np.linspace(0.7, 0, 60, endpoint=False),
        rf_frequencies_hz=(2e9, 2.25e9),
        rf_powers_dbm=(4.0,), lockin_frequencies_hz=(17.37,),
    )
    print('Saved:', combined_scan.run_experiment())

## 5 — Angular dependence (original `new_ad_angle_sweep`, `extra_angle_sweep`, `angle_sweep_full`)

`angles_deg` lists **relative** sample angles from the currently positioned stage. The class converts each angle to integer motor steps using the original `380/180` calibration and moves the motor by the difference. It does **not** home or return to zero automatically. Physical direction 0 vs 1 must be verified for the actual wiring.

A forward and reverse branch should be executed as distinct runs after verifying the sample's starting orientation. This notebook uses an example 0°–90° sweep; do not assume the sample is automatically returned.

In [ ]:
angle_values = np.arange(0, 100, 10)  # relative degrees from initial stage orientation
if RUN_SCANS:
    angular_field = make_stfmr(fields=FIELD_VALUES, angles_deg=angle_values,
                               rf_frequencies_hz=(2.5e9,), rf_powers_dbm=(12.0,))
    print('Saved:', angular_field.run_experiment())

# To reproduce a second, reverse-angle pass: reposition the stage first and
# use the appropriate physical orientation/direction calibration.

## 6 — Read and plot measured data

Uses PIEC's standard metadata + blank row + data table format (CSV), like the AMR notebook. The backend also saves a PNG. No ST-FMR Lorentzian fit is inferred from the source notebook.

In [ ]:
if RUN_SCANS and field_experiment.filename and Path(field_experiment.filename).exists():
    metadata, results = load_stfmr_data(field_experiment.filename)
    display(metadata.T)
    display(results.head())
    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    axes[0].plot(results['field'], results['X'], 'o-')
    axes[0].set_ylabel('Lock-in X (V)')
    axes[1].plot(results['field'], results['Y'], 'o-')
    axes[1].set_xlabel(f'Field ({FIELD_UNIT})')
    axes[1].set_ylabel('Lock-in Y (V)')
    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No data has been acquired in this notebook session.')

## 7 — Cleanup and physical-hardware checklist

- Check calibration workbook covers the complete field range, including negative field where used.
- Verify DAQ model, output channel 0, input channel 0, supply limits, motor 380-step/180° conversion, and direction bits.
- Confirm RF model, command manual, output power limits, and that the RF connector actually reaches the sample.
- Do not interrupt DAQ communication while outputs are enabled. Even after a software shutdown, **inspect the physical supply/RF state before leaving the system**.

The cell below issues best-effort output-off commands only; it never homes the motor.

In [ ]:
# Explicit cleanup. Safe to run repeatedly after virtual or physical testing.
if USE_VIRTUAL or RUN_SCANS:
    try:
        daq.write_AO(0, 0.0)
    finally:
        rf_source.output(1, on=False)
    print('DAQ analog output commanded to 0 V; RF commanded OFF.')
else:
    print('Physical scan was not enabled; confirm instrument outputs locally.')